In [ ]:
!pip install stable-baselines3 transformers sentence-transformers rouge-score nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 94.4 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback

class RewardCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []

    def _on_step(self) -> bool:
        # Access the reward from the environment
        if 'rewards' in self.locals:
            reward = self.locals['rewards']
            if isinstance(reward, (list, np.ndarray)):
                reward = reward[0]
            self.episode_rewards.append(reward)
        return True

In [ ]:
!pip install "shimmy>=2.0"

In [ ]:
# -*- coding: utf-8 -*-
"""Enhanced Multi-Agent Summarizer using A2C with dynamic extraction count for better content coverage."""

# ---------------- Imports & Setup ----------------
import os
import torch
import numpy as np
import gym
import random
from gym import spaces
from sklearn.metrics.pairwise import cosine_similarity
from transformers import T5Tokenizer, T5ForConditionalGeneration, BartTokenizer, BartForConditionalGeneration
from sentence_transformers import SentenceTransformer
from stable_baselines3 import A2C
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
# ---------------- Agent 1: Sentence Extraction with A2C ----------------
class SentenceExtractionEnv(gym.Env):
    def __init__(self, sentences, embeddings):
        super().__init__()
        self.sentences = sentences
        self.embeddings = embeddings
        self.num_sentences = len(sentences)
        self.current_idx = 0
        self.selected = []

        dim = embeddings.shape[1]
        self.action_space = spaces.Discrete(2)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(dim,), dtype=np.float32)

    def reset(self):
        self.current_idx = 0
        self.selected = []
        return self.embeddings[self.current_idx]

    def step(self, action):
        if action == 1:
            self.selected.append((self.sentences[self.current_idx], self.embeddings[self.current_idx]))
        self.current_idx += 1
        done = self.current_idx >= self.num_sentences

        if done:
            if len(self.selected) < 3:
                return np.zeros_like(self.embeddings[0]), -1.0, True, {}
            selected_text = ' '.join([s for s, _ in self.selected])
            ref_text = ' '.join(self.sentences)
            scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True)
            r1 = scorer.score(ref_text, selected_text)['rouge1'].fmeasure
            r2 = scorer.score(ref_text, selected_text)['rouge2'].fmeasure
            score = (r1 + r2) / 2
            return np.mean([e for _, e in self.selected], axis=0), score, True, {}

        return self.embeddings[self.current_idx], 0.0, False, {}

class SentenceExtractor:
    def __init__(self, model_path='a2c_extractor.zip'):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.encoder = SentenceTransformer('all-mpnet-base-v2')
        self.model_path = model_path
        self.model = None

    def encode(self, sentences):
        return self.encoder.encode(sentences, convert_to_numpy=True)

    def train_and_extract(self, sentences, timesteps=15000):
        embeddings = self.encode(sentences)
        env = SentenceExtractionEnv(sentences, embeddings)

        if os.path.exists(self.model_path):
            os.remove(self.model_path)

        self.model = A2C('MlpPolicy', env, verbose=0)
        self.model.learn(total_timesteps=timesteps)
        self.model.save(self.model_path)

        obs = env.reset()
        selected = []
        for _ in range(len(sentences)):
            action, _ = self.model.predict(obs)
            if env.current_idx < len(sentences) and action == 1:
                selected.append(sentences[env.current_idx])
            obs, _, done, _ = env.step(action)
            if done:
                break

        if len(selected) < 4:
            selected = sentences[:min(6, len(sentences))]  # fallback: select more content

        return ' '.join(selected)

In [ ]:
# ---------------- Agent 2: T5-Based Text Simplifier ----------------
class TextSimplifier:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = T5Tokenizer.from_pretrained('t5-base')
        self.model = T5ForConditionalGeneration.from_pretrained('t5-base').to(self.device)

    def simplify(self, text):
        input_text = "paraphrase: " + text
        inputs = self.tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(self.device)
        with torch.no_grad():
            output = self.model.generate(
                inputs.input_ids,
                max_length=300,
                num_beams=6,
                temperature=0.8,
                top_p=0.9,
                no_repeat_ngram_size=2,
                early_stopping=True
            )
        return self.tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# ---------------- Agent 3: BART-A2C-Based Abstractive Synthesizer ----------------
class SummarySynthesizer:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
        self.model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn').to(self.device)

    def summarize(self, input_text):
        inputs = self.tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024, padding="longest").to(self.device)
        with torch.no_grad():
            output = self.model.generate(
                inputs.input_ids,
                max_length=256,
                min_length=120,
                num_beams=8,
                temperature=0.7,
                top_k=50,
                top_p=0.95,
                length_penalty=1.0,
                no_repeat_ngram_size=3,
                early_stopping=True
            )
        return self.tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# ---------------- Main Multi-Agent Summarizer Pipeline ----------------
def multi_agent_summarizer(text):
    sentences = sent_tokenize(text)
    if len(sentences) < 5:
        print("[Short text] Skipping RL extractor. Summarizing directly.")
        synthesizer = SummarySynthesizer()
        summary = synthesizer.summarize(text)
        #rouge = compute_rouge(text, summary)
        #print("[ROUGE Scores]", rouge)
        return summary

    extractor = SentenceExtractor()
    simplifier = TextSimplifier()
    synthesizer = SummarySynthesizer()

    #print("[Agent 1] Extracting Important Sentences via A2C...")
    extracted = extractor.train_and_extract(sentences)
    #print(f"[Extracted] {extracted}\n")

    #print("[Agent 2] Simplifying Extracted Sentences...")
    simplified = simplifier.simplify(extracted)
    #print(f"[Simplified] {simplified}\n")

    if not simplified or len(simplified.split()) < 5:
        #print("[Fallback] Simplified content too short. Using extracted.")
        simplified = extracted

    #print("[Agent 3] Generating Final Abstractive Summary...")
    summary = synthesizer.summarize(simplified)

    #print("[ROUGE Scores]")
    #rouge = compute_rouge(text, summary)
    #print(rouge)

    return summary

# ---------------- Example Demo Run ----------------
if __name__ == "__main__":
    article = """
    NASA's Aquarius was the primary instrument on the SAC-D spacecraft. It consisted of three passive microwave radiometers to detect the surface emission that was used to obtain salinity and an active scatterometer to measure the ocean waves that affected the precision of the salinity measurement. While salinity levels in the open ocean generally range from 32 to 37 practical salinity units, or psu (roughly equivalent to parts per thousand), the Aquarius sensor was able to detect changes in salinity as small as 0.2 psu. This is equivalent to about a "pinch" (i.e., 1/8 of a teaspoon) of salt in one gallon of water.
Aquarius/SAC-D was a collaboration between NASA and Argentina's space agency, Comision Nacional de Actividades Espaciales (CONAE), with participation from Brazil, Canada, France and Italy. The Aquarius instrument was jointly built by NASA's Jet Propulsion Laboratory, Pasadena, California, and NASA's Goddard Space Flight Center, Greenbelt, Maryland. JPL managed Aquarius through the mission's commissioning phase and archived mission data. Goddard managed the mission's operations phase and processed Aquarius science data. NASA's Launch Services Program at the Kennedy Space Center in Florida is managing the launch. CONAE provided the SAC-D spacecraft, an optical camera, a thermal camera in collaboration with Canada, a microwave radiometer, sensors developed by various Argentine institutions, and the mission operations center in Argentina. France and Italy also contributed instruments.
    """
    summary = multi_agent_summarizer(article)
    print("\n[Final Summary]\n", summary)



[Final Summary]
 Aquarius/SAC-D was a collaboration between NASA and Argentina's space agency, Comision Nacional de Actividades Espaciales (CONAE) The Aquarius instrument was jointly built by NASA's Jet Propulsion Laboratory, Pasadena, California, and NASA's Goddard Space Flight Center, Greenbelt, Maryland. JPL managed Aquarius through the mission's commissioning phase and archived mission data. Goddard managed the mission’s operations phase and processed Aquarius science data. NASA's Launch Services Program at the Kennedy Space Center, Florida, is managing the launch.


#Deployment

In [ ]:
# Install Gradio
!pip install gradio --quiet

# Import Gradio
import gradio as gr

# Wrap your summarization function if needed
def summarize(text):
    return multi_agent_summarizer(text)

# Define the Gradio interface
demo = gr.Interface(
    fn=summarize,
    inputs=gr.Textbox(lines=15, placeholder="Paste article here...", label="Input Article"),
    outputs=gr.Textbox(label="Summarized Output"),
    title="Multi-Agent Text Summarizer (MATSRL)",
    description="⚡ Try our live demo of a multi-agent summarizer that combines RL and transformer-based models."
)

# Launch the app and share link
demo.launch(share=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.8 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/websockets/legacy/__init__.py:6: DeprecationWarning: websockets.legacy is deprecated; see https://websockets.readthedocs.io/en/stable/howto/upgrade.html for upgrade instructions
  warnings.warn(  # deprecated in 14.0 - 2024-11-09
/usr/local/lib/python3.11/dist-packages/uvicorn/protocols/websockets/websockets_impl.py:17: DeprecationWarning: websockets.server.WebSocketServerProtocol is deprecated
  from websockets.server import WebSocketServerProtocol


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a0b49278193c6b450.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
